In [2]:
!pip install -q ultralytics

In [3]:
import ultralytics 
ultralytics.checks()

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 5642.4/8156.9 GB disk)


In [4]:
from ultralytics.data.utils import check_det_dataset
data = check_det_dataset("VisDrone.yaml")
print(data['train'])
print(data['names'])


WARNING ⚠️ Dataset 'VisDrone.yaml' images not found, missing path '/kaggle/working/datasets/VisDrone/images/val'
Unzipping /kaggle/working/datasets/VisDrone/VisDrone2019-DET-val.zip to /kaggle/working/datasets/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 2.2Kfiles/s 0.5s0.0sB/s 2.6s<3.9s
Unzipping /kaggle/working/datasets/VisDrone/VisDrone2019-DET-test-dev.zip to /kaggle/working/datasets/VisDrone/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 1.8Kfiles/s 1.8s0.1s
Unzipping /kaggle/working/datasets/VisDrone/VisDrone2019-DET-train.zip to /kaggle/working/datasets/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 2.2Kfiles/s 5.9s<0.0s
Converting train: ━━━━━━━━━━━━ 6471 2.2Kit/s 2.8s
Converting val: ━━━━━━━━━━━━ 548 1.4Kit/s 0.3ss
Converting test: ━━━━━━━━━━━━ 1610 2.2Kit/s 0.7s
Dataset download success ✅ (49.0s), saved to /kaggle/working/datasets

/kaggle/working/datasets/VisDrone/images/train
{0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'ca

In [5]:
from pathlib import Path
from PIL import Image
from ultralytics.data.utils import img2label_paths
import numpy as np, collections

imgs = sorted(Path(data["train"]).glob("*.jpg"))
lbls = img2label_paths([str(p) for p in imgs])
names = data["names"]

class_counts = collections.Counter()
areas, per_img = [], []

for ip, lp in zip(imgs, lbls):
    lp = Path(lp)
    if not lp.exists():
        per_img.append(0)
        continue
    w, h = Image.open(ip).size
    rows = np.loadtxt(lp, ndmin=2)
    per_img.append(len(rows))
    for c, x, y, bw, bh in rows:
        class_counts[int(c)] += 1
        areas.append(bw * w * bh * h)

areas = np.array(areas)
print(f"Images: {len(imgs)} | Boxes: {len(areas)} | Avg boxes/img: {np.mean(per_img):.1f}")
print(f"Small  (<32²px):  {np.mean(areas < 32**2):.1%}")
print(f"Medium (32²–96²): {np.mean((areas >= 32**2) & (areas < 96**2)):.1%}")
print(f"Large  (>96²px):  {np.mean(areas >= 96**2):.1%}")
for c, n in class_counts.most_common():
    print(f"{names[c]:>16}: {n}")

Images: 6471 | Boxes: 343205 | Avg boxes/img: 53.0
Small  (<32²px):  60.5%
Medium (32²–96²): 34.0%
Large  (>96²px):  5.5%
             car: 144867
      pedestrian: 79337
           motor: 29647
          people: 27059
             van: 24956
           truck: 12875
         bicycle: 10480
             bus: 5926
        tricycle: 4812
 awning-tricycle: 3246


In [6]:
print("Images found:", len(imgs))
print("Label files found:", sum(Path(l).exists() for l in lbls))
print("Boxes collected:", len(areas))

Images found: 6471
Label files found: 6471
Boxes collected: 343205


In [11]:
#Model training 
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

results = model.train(
    data = "VisDrone.yaml",
    epochs = 30,
    imgsz = 640,
    batch = 16,
    device = 0,
    seed = 0,
    project = "/kaggle/working/runs",
    name = "yolo11n_640",
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_640-2, nbs=64, nms=None, opset=None, optimize=False, optimizer=aut

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      6.95G      1.798      1.412     0.9639        556        640: 100% ━━━━━━━━━━━━ 405/405 6.5it/s 1:020.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 4.0it/s 4.5s0.3s
                   all        548      38759      0.261      0.211      0.164     0.0851

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      6.95G      1.767      1.356     0.9533        628        640: 100% ━━━━━━━━━━━━ 405/405 6.6it/s 1:010.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 4.1it/s 4.4s0.2s
                   all        548      38759      0.295      0.247      0.202      0.104

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      6.95G      1.748       1.33     0.9421        473        640: 100% ━━━━━━━━━━━━ 405/405 6.6it/s 1:020.1sss
                 Class     Ima

In [12]:
!cd /kaggle/working/runs && zip -r yolo11n_640.zip yolo11n_640-2

  adding: yolo11n_640-2/ (stored 0%)
  adding: yolo11n_640-2/train_batch8102.jpg (deflated 6%)
  adding: yolo11n_640-2/val_batch1_labels.jpg (deflated 5%)
  adding: yolo11n_640-2/BoxPR_curve.png (deflated 5%)
  adding: yolo11n_640-2/val_batch1_pred.jpg (deflated 4%)
  adding: yolo11n_640-2/train_batch2.jpg (deflated 2%)
  adding: yolo11n_640-2/BoxP_curve.png (deflated 4%)
  adding: yolo11n_640-2/labels.jpg (deflated 27%)
  adding: yolo11n_640-2/results.csv (deflated 61%)
  adding: yolo11n_640-2/train_batch8100.jpg (deflated 10%)
  adding: yolo11n_640-2/train_batch0.jpg (deflated 3%)
  adding: yolo11n_640-2/weights/ (stored 0%)
  adding: yolo11n_640-2/weights/last.pt (deflated 10%)
  adding: yolo11n_640-2/weights/best.pt (deflated 10%)
  adding: yolo11n_640-2/results.png (deflated 8%)
  adding: yolo11n_640-2/val_batch2_pred.jpg (deflated 5%)
  adding: yolo11n_640-2/val_batch2_labels.jpg (deflated 5%)
  adding: yolo11n_640-2/confusion_matrix_normalized.png (deflated 14%)
  adding: yolo11

In [13]:
!ls -lh /kaggle/working/runs/*.zip

-rw-r--r-- 1 root root 19M Nov 21 13:21 /kaggle/working/runs/yolo11n_640.zip


In [14]:
import os
from IPython.display import FileLink
os.chdir("/kaggle/working")
FileLink("runs/yolo11n_640.zip")

/kaggle/working/runs/yolo11n_640.zip